# ED Fig 3g — FOXA2 and SOX2 versus FOXF1 pixel density, day 2 cysts

**Feeds:** **ED Fig 3g**, two panels: FOXA2 versus FOXF1 on top, SOX2 versus FOXF1 on the
bottom. Section 6 writes the per-pixel table to `derived/`. `scripts/render_ed3g_panel.py`
redraws both panels from that table, with no images.

Ported from `day2 analysis(FOXA2_SOX2)-20260916.ipynb`, by T.H., which pools seven day-2 cysts
from three independent differentiations (folder prefixes 1.3, 1.5, 1.7), each delineated
manually in brightfield with the bead excluded. The recipe below is unchanged: the same
per-cyst background offsets, the same `SEED`/`JITTER` dither, the same positivity gates and the
same quadrant statistics, computed from the undithered values.

**Changes from the original notebook**

1. The seven cyst crop folders are read from `data/cropped2/` under this lane, or from
   `ED3G_DATA_DIR`, instead of the original's `E:\Supplementary figures\Section 2\day2
   analysis\cropped2`; a missing folder now says so rather than failing partway through the
   first cyst loop.
2. Panel PDFs (section 4) and the per-pixel CSV (section 6) are written to `output/` beside
   this notebook, not back into the original folder tree.
3. Stored outputs were cleared; they carried the original machine's `E:\` paths.

Nothing about the recipe itself changed: same `SUBBACKGROUND`/`INCLUDE_BEAD` settings, same
per-cyst offsets in `CYSTS`, same gates, same `HIST_BINS`.

**Not in this repository:** the seven cyst crop folders, each holding `select_bead.png`
(the manual cyst/bead ROI mask), `FOXA2.png`, `SOX2.png` and `FOXF1.png`. Raw imaging is
available from the lead contact on request. Everything the panels need is in `derived/`.


In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import colors
import scipy.ndimage

# The seven cyst crop folders live under this lane's data/cropped2/ unless ED3G_DATA_DIR says
# otherwise (pointed at a directory that itself contains cropped2/).
BASE = os.environ.get("ED3G_DATA_DIR", os.path.join(os.path.dirname(os.getcwd()), "data"))
CROP = os.path.join(BASE, "cropped2")

if not os.path.isdir(CROP):
    raise SystemExit(
        f"{CROP} is not there. This notebook reads seven cyst crop folders (select_bead.png, "
        f"FOXA2.png, SOX2.png and FOXF1.png in each), which are not in this repository; the "
        f"panels themselves are redrawn from derived/ by scripts/render_ed3g_panel.py.")

SEED          = 0        # dither seed
JITTER        = 0.5      # +/- half a quantization step (8-bit data)
SUBBACKGROUND = "clamp"  # "keep" | "clamp" (negatives -> 0) | "drop"
INCLUDE_BEAD  = False    # True = keep the bead inside the analysis region

GATE_X = 15.0            # positivity gate, second marker
GATE_Y = {"FOXA2": 45.0, "SOX2": 45.0}   # positivity gate, FOXF1, same for both panels
                         # Both gates are in background-subtracted 8-bit units, that is,
                         # grey levels above the per-cyst offsets set in CYSTS below.
                         # GATE_Y = 45 therefore corresponds to a raw value of 55, or 65
                         # for cyst 1.7, whose FOXF1 offset is 20.

HIST_BINS      = 256
CONTOUR_LEVELS = np.linspace(0.05, 1, 12)
SMOOTH_SIGMA   = (2, 1)
CLIP_PCT       = 99      # colour-scale reference percentile, see density_panel

# Folder, then the background offsets for each panel as
# (offset on the second marker, offset on FOXF1), in raw 8-bit units.
# The first pair is used for the FOXA2 panel, the second for the SOX2 panel.
# Offsets were set by hand per image and per channel, since the background level differs
# between channels and between acquisitions. For a given cyst the FOXF1 offset is the same
# in both panels, since it is the same image.
CYSTS = [
    ("1.5",                        (15, 10), (20, 10)),
    (os.path.join("1.5", "1.5.2"), (10, 10), (20, 10)),
    (os.path.join("1.3", "1.3.1"), (15, 10), (20, 10)),
    (os.path.join("1.3", "1.3.2"), (15, 10), (20, 10)),
    (os.path.join("1.3", "1.3.3"), (15, 10), (20, 10)),
    (os.path.join("1.3", "1.3.4"), (15, 10), (20, 10)),
    ("1.7",                        (25, 20), (20, 20)),
]

print("cysts:", len(CYSTS), "| SUBBACKGROUND =", SUBBACKGROUND,
      "| INCLUDE_BEAD =", INCLUDE_BEAD)


## 1. Images and cyst region


In [ ]:
RED_LO, RED_HI = np.array([150, 0, 0]), np.array([255, 50, 50])
GRN_LO, GRN_HI = np.array([0, 200, 0]), np.array([100, 255, 100])


def find(d, name):
    for f in os.listdir(d):
        if f.lower() == name.lower():
            return os.path.join(d, f)
    raise FileNotFoundError(os.path.join(d, name))


def load_gray(path):
    return np.array(Image.open(path).convert("L")).astype(float)


def cyst_region(d):
    """Red = cyst, green = bead."""
    rgb = np.array(Image.open(find(d, "select_bead.png")).convert("RGB"))
    red = np.all((rgb >= RED_LO) & (rgb <= RED_HI), axis=-1)
    bead = np.all((rgb >= GRN_LO) & (rgb <= GRN_HI), axis=-1)
    return (red | bead) if INCLUDE_BEAD else (red & ~bead), bead


rows = []
for name, _, _ in CYSTS:
    d = os.path.join(CROP, name)
    roi, bead = cyst_region(d)
    rows.append({"cyst": name, "region px": int(roi.sum()), "bead px": int(bead.sum())})
print(pd.DataFrame(rows).to_string(index=False))


## 2. Build the pooled arrays

A constant background offset is subtracted from each channel, values below zero are set to
zero, and each channel is normalized to its maximum over the pooled pixels, so that each
axis spans 0 to 1.


In [ ]:
rng = np.random.default_rng(SEED)


def build_panel(marker, which):
    """which = 0 for the FOXA2 panel, 1 for the SOX2 panel."""
    xs, ys, per_cyst = [], [], []
    for name, pars0, pars1 in CYSTS:
        off_marker, off_foxf1 = (pars0, pars1)[which]
        d = os.path.join(CROP, name)
        roi, _ = cyst_region(d)
        x = load_gray(find(d, marker + ".png"))[roi] - off_marker
        y = load_gray(find(d, "FOXF1.png"))[roi] - off_foxf1

        if SUBBACKGROUND == "clamp":
            x, y = np.maximum(x, 0), np.maximum(y, 0)
        elif SUBBACKGROUND == "drop":
            keep = (x > 0) & (y > 0)
            x, y = x[keep], y[keep]

        xs.append(x)
        ys.append(y)
        per_cyst.append({"cyst": name, "pixels": int(x.size)})

    # each channel divided by its maximum over the pooled pixels
    x = np.concatenate(xs)
    y = np.concatenate(ys)
    XSCALE, YSCALE = float(x.max()), float(y.max())
    x, y = x / XSCALE, y / YSCALE
    xd = x + rng.uniform(-JITTER, JITTER, size=x.shape) / XSCALE
    yd = y + rng.uniform(-JITTER, JITTER, size=y.shape) / YSCALE
    if SUBBACKGROUND == "clamp":
        # keep the dithered copy on the same support as the statistics
        xd, yd = np.clip(xd, 0, 1), np.clip(yd, 0, 1)
    return dict(x=x, y=y, xd=xd, yd=yd, xscale=XSCALE, yscale=YSCALE,
                table=pd.DataFrame(per_cyst))


PANELS = {}
for i, marker in enumerate(["FOXA2", "SOX2"]):
    PANELS[marker] = build_panel(marker, i)
    P = PANELS[marker]
    print(f"--- {marker} panel ---")
    print(P["table"].to_string(index=False))
    print(f"  pooled n = {P['x'].size}   "
          f"{marker} [{P['x'].min():.3f}, {P['x'].max():.3f}]   "
          f"FOXF1 [{P['y'].min():.3f}, {P['y'].max():.3f}]")
    print(f"  at or below background: {marker} {(P['x'] == 0).mean():.1%}, "
          f"FOXF1 {(P['y'] == 0).mean():.1%}")
    print()


## 3. Quadrant and exclusivity statistics


In [ ]:
def stats(x, y, xg, yg):
    xp, yp = x >= xg, y >= yg
    Qll = int(np.sum(~xp & ~yp)); Qlr = int(np.sum(xp & ~yp))
    Qul = int(np.sum(~xp & yp));  Qur = int(np.sum(xp & yp))
    N = Qll + Qlr + Qul + Qur
    either = Qlr + Qul + Qur
    nan = float("nan")
    return {
        "Qll": Qll / N, "Qlr": Qlr / N, "Qul": Qul / N, "Qur": Qur / N,
        "n_total": N, "n_either": either, "n_double": Qur,
        "n_marker_only": Qlr, "n_FOXF1_only": Qul,
        "Jaccard": Qur / either if either else nan,
        "P_FOXF1_given_marker": Qur / (Qur + Qlr) if (Qur + Qlr) else nan,
        "P_marker_given_FOXF1": Qur / (Qur + Qul) if (Qur + Qul) else nan,
    }


for marker, P in PANELS.items():
    xg, yg = GATE_X / P["xscale"], GATE_Y[marker] / P["yscale"]
    P["xg"], P["yg"] = xg, yg
    P["stats"] = stats(P["x"], P["y"], xg, yg)
    s = P["stats"]
    print(f"--- {marker} vs FOXF1  (gates: {marker} {GATE_X:.0f}, FOXF1 {GATE_Y[marker]:.0f} raw"
          f" -> {xg:.4f}, {yg:.4f} normalized; scales {P['xscale']:.0f}, {P['yscale']:.0f}) ---")
    print(f"  Qll {s['Qll']:.4f}  Qlr {s['Qlr']:.4f}  Qul {s['Qul']:.4f}  Qur {s['Qur']:.4f}"
          f"   (n = {s['n_total']})")
    print(f"  Jaccard (double+ / either)   {s['Jaccard'] * 100:.2f}%")
    print(f"  P(FOXF1+ | {marker}+)        {s['P_FOXF1_given_marker'] * 100:.2f}%")
    print(f"  P({marker}+ | FOXF1+)        {s['P_marker_given_FOXF1'] * 100:.2f}%")
    print()


## 4. Density plots

Axis limits cover every analysed pixel. The dense band along each axis corresponds to
pixels with no signal above background in that channel.


In [ ]:
CMAPS = {
    "FOXA2": ["#FFFFFF", "#F4C4F3", "#E052D0", "#9D0191"],
    "SOX2":  ["#FFFFFF", "#FFF7B2", "#FFE600", "#FFC300"],
}

# Written beside the notebook, not back into BASE.
OUT_FIG = os.path.join(os.path.dirname(os.getcwd()), "output", "figures")
os.makedirs(OUT_FIG, exist_ok=True)


def density_panel(marker, save=True, annotate=True):
    P = PANELS[marker]
    xs, ys = P["xd"], P["yd"]

    H, xe, ye = np.histogram2d(xs, ys, bins=HIST_BINS)
    H_norm = np.log1p(H) / np.log1p(H).max()
    H_smooth = scipy.ndimage.gaussian_filter(H_norm, sigma=SMOOTH_SIGMA)

    # Colour scale. Dividing by the maximum leaves almost the whole panel pale,
    # because the single bin at the origin is far denser than everything else. The
    # fill is therefore scaled to the CLIP_PCT-th percentile of bin density and
    # clipped, so that 1 on the colour bar means "at or above that percentile".
    # The contours are drawn from H_norm and are unaffected.
    H_disp = np.clip(H_norm / np.percentile(H_norm, CLIP_PCT), 0, 1)
    cmap = colors.LinearSegmentedColormap.from_list("c", CMAPS[marker])

    plt.style.use("default")
    fig, ax = plt.subplots(facecolor="white")
    im = ax.imshow(H_disp.T, origin="lower", cmap=cmap,
                   extent=[xe[0], xe[-1], ye[0], ye[-1]],
                   aspect="auto", vmin=0, vmax=1)
    ax.contour(0.5 * (xe[:-1] + xe[1:]), 0.5 * (ye[:-1] + ye[1:]), H_smooth.T,
               levels=CONTOUR_LEVELS, colors="black", linewidths=0.5)

    xg, yg = P["xg"], P["yg"]
    ax.axvline(xg, color="0.35", ls="--", lw=0.8)
    ax.axhline(yg, color="0.35", ls="--", lw=0.8)

    xlo, xhi = float(xs.min()), float(xs.max())
    ylo, yhi = float(ys.min()), float(ys.max())
    if annotate:
        s = P["stats"]
        for tx, ty, key in [(xlo, ylo, "Qll"), (xhi, ylo, "Qlr"),
                            (xlo, yhi, "Qul"), (xhi, yhi, "Qur")]:
            ax.annotate(f"{s[key] * 100:.1f}%", xy=((tx + xg) / 2, (ty + yg) / 2),
                        ha="center", va="center", fontsize=9, weight="bold")

    ax.set_xlim(xlo, xhi)
    ax.set_ylim(ylo, yhi)
    ax.set_xlabel(marker + " normalized intensity")
    ax.set_ylabel("FOXF1 normalized intensity")
    ax.set_title(marker + " vs FOXF1 pixel density")
    fig.colorbar(im, ax=ax, label="Pixel density")
    plt.tight_layout()

    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    if save:
        tag = "" if SUBBACKGROUND == "clamp" else "_" + SUBBACKGROUND
        out = os.path.join(OUT_FIG, "ED3g_" + marker + "_vs_FOXF1" + tag + ".pdf")
        plt.savefig(out, format="pdf", dpi=300)
        print("saved:", out)
    plt.show()


density_panel("FOXA2")


In [ ]:
density_panel("SOX2")


## 5. Effect of the sub-background setting

Rebuilds both panels for each setting of `SUBBACKGROUND`, using the same gates as the
figure, and prints the numbers side by side without touching `PANELS`. The `clamp` rows
therefore reproduce the reported values. Dropping pixels below background keeps only
pixels above background in both channels, which conditions on the co-positivity the plot
is meant to assess, so the double-positive share rises.


In [ ]:
def compare_switch():
    global SUBBACKGROUND, rng
    setting = SUBBACKGROUND      # restored on the way out
    out = []
    for flag in ("keep", "clamp", "drop"):
        SUBBACKGROUND = flag
        rng = np.random.default_rng(SEED)
        for i, marker in enumerate(["FOXA2", "SOX2"]):
            P = build_panel(marker, i)
            s = stats(P["x"], P["y"], GATE_X / P["xscale"],
                      GATE_Y[marker] / P["yscale"])
            out.append({"sub-bg": flag, "panel": marker, "n": s["n_total"],
                        "Qll%": s["Qll"] * 100, "Qlr%": s["Qlr"] * 100,
                        "Qul%": s["Qul"] * 100, "Qur%": s["Qur"] * 100,
                        "Jaccard%": s["Jaccard"] * 100,
                        "P(FOXF1+|m+)%": s["P_FOXF1_given_marker"] * 100,
                        "P(m+|FOXF1+)%": s["P_marker_given_FOXF1"] * 100})
    SUBBACKGROUND = setting
    rng = np.random.default_rng(SEED)
    return pd.DataFrame(out)


print(compare_switch().to_string(index=False, float_format=lambda v: f"{v:8.2f}"))


## 6. Source data

One row per pixel, in the same column order as the sheet in
`Source Data Extended Data Fig.3.xlsx`. Written beside this notebook, not over the shipped
`derived/ed3g_per_pixel.tsv`.


In [ ]:
# The two panels analyse the same pixels, so both coordinate pairs sit side by side.
OUT = os.path.join(os.path.dirname(os.getcwd()), "output", "derived")
os.makedirs(OUT, exist_ok=True)

out = pd.DataFrame({
    "FOXA2 normalized intensity": PANELS["FOXA2"]["x"].round(6),
    "FOXF1 normalized intensity (FOXA2 panel)": PANELS["FOXA2"]["y"].round(6),
    "SOX2 normalized intensity":  PANELS["SOX2"]["x"].round(6),
    "FOXF1 normalized intensity (SOX2 panel)":  PANELS["SOX2"]["y"].round(6),
})
out.to_csv(os.path.join(OUT, "ED3g_per_pixel.csv"), index=False)
print(f"{len(out)} rows written to ED3g_per_pixel.csv")
for marker, P in PANELS.items():
    s = P["stats"]
    print(f"  {marker}: Qll {s['Qll']*100:.2f}  Qlr {s['Qlr']*100:.2f}  Qul {s['Qul']*100:.2f}"
          f"  Qur {s['Qur']*100:.2f}   Jaccard {s['Jaccard']*100:.2f}%")


## 7. Numbers for the legend


In [ ]:
for marker, P in PANELS.items():
    s = P["stats"]
    n_marker = s["n_marker_only"] + s["n_double"]
    n_foxf1  = s["n_FOXF1_only"] + s["n_double"]
    n_neg    = s["n_total"] - s["n_either"]
    print(f"{marker} vs FOXF1:  n = {len(CYSTS)} cysts, {s['n_total']} pixels analysed")
    print("   quadrant proportions, as annotated on the panel:")
    for corner, label, frac, count in [
            ("lower left ", "double-negative",         s["Qll"], n_neg),
            ("lower right", f"{marker}+ FOXF1-",       s["Qlr"], s["n_marker_only"]),
            ("upper left ", f"{marker}- FOXF1+",       s["Qul"], s["n_FOXF1_only"]),
            ("upper right", "double-positive",         s["Qur"], s["n_double"])]:
        print(f"     {corner}   {label:<16}{frac * 100:5.1f}%   ({count} px)")
    print("   derived from the same gates:")
    print(f"     {marker}+ {n_marker}   FOXF1+ {n_foxf1}   either {s['n_either']}")
    print(f"     double-positive = {s['Jaccard'] * 100:.2f}% of pixels positive for either"
          f" marker (Jaccard index)")
    print(f"     {s['P_marker_given_FOXF1'] * 100:.2f}% of FOXF1+ pixels are {marker}+")
    print(f"     {s['P_FOXF1_given_marker'] * 100:.2f}% of {marker}+ pixels are FOXF1+")
    print()


## 8. Gate sensitivity

The quadrant percentages depend on where the gates are placed, so the scan below varies
both gates around the values used in the figure. The marker gate barely matters; the FOXF1
gate does. Across the whole range the double-positive share stays small, so the
conclusion does not rest on the particular values chosen.


In [ ]:
# The gates do not enter the density map, the contours or the source data. The values used
# in the figure are marked with an asterisk.
def gate_scan(marker, xgs, ygs):
    P = PANELS[marker]
    rows = []
    for gy in ygs:
        for gx in xgs:
            s = stats(P["x"], P["y"], gx / P["xscale"], gy / P["yscale"])
            used = gx == GATE_X and gy == GATE_Y[marker]
            rows.append({"FOXF1 gate": gy, marker + " gate": f"{gx:g}" + (" *" if used else ""),
                         "double+ %": s["Qur"] * 100, "Jaccard %": s["Jaccard"] * 100,
                         f"P(FOXF1+|{marker}+) %": s["P_FOXF1_given_marker"] * 100})
    return pd.DataFrame(rows)


for marker in PANELS:
    print(f"--- {marker} vs FOXF1: gate sensitivity ---")
    print(gate_scan(marker, [10, 15, 20], [35, 45, 55]).to_string(
        index=False, float_format=lambda v: f"{v:7.2f}"))
    print()
